In [ ]:
import random, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from datetime import date, datetime, timedelta
from matplotlib.colors import ListedColormap, BoundaryNorm
sns.set_theme(style="ticks")
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'

# parameters

In [ ]:
yeari, yearf = '2024', '2024'
weeki, weekf = '18', '31'

In [ ]:
di = datetime.strptime(f'{yeari}-{weeki}-1', "%Y-%W-%w").date()
df = datetime.strptime(f'{yearf}-{weekf}-1', "%Y-%W-%w").date() + timedelta(6)
ds = [di+timedelta(dt) for dt in range((df-di).days+1)]
daylist = ds
print(di, 'until', df)

In [ ]:
cdef = 'tl7_10m'# 'tl5_10m' 'tl6_10m' 'tl7_10m' 'tl8_10m' 'tl8_60m'
cdef_alt = '16m_10min'# tl5: 62 ... tl7: 16   tl8: 8

# load data

In [ ]:
data = pd.read_csv('data/fig6/follow_didpairs.csv')
data['day'] = [d.date() for d in pd.to_datetime(data.day)]
data['stime'] = pd.to_datetime(data.stime)
dmin = data.day.min()
data['tt2'] = data.day.apply(lambda d: (d-dmin).days)*24*6 + data.stime.dt.hour*6 + (data.stime.dt.minute//10)
print(data.tt2.min(), data.tt2.max())

In [ ]:
(data.day.max()-dmin).days*24*6 + (24-1)*6 + (6-1)

# analyses

In [ ]:
# stability: 0 = random pair, 1 = stable pair
randstab = pd.DataFrame(data.groupby('pair').day.apply(lambda x: 0 if len(set(x))==1 else 1)).reset_index()
randstab = randstab.rename(columns={'tt':'stability','day':'stability','tt2':'stability'})
randstab

In [ ]:
for_mat = data[['day','pair']].drop_duplicates()#.merge(randstab, on='pair', how='left')
for_mat['on'] = 1
for_mat = for_mat.set_index(['day','pair']).unstack('day').fillna(0).astype(int).sort_values(by='day', axis=1)
for_mat = for_mat['on'].reset_index().merge(randstab, on='pair').set_index('pair')
for_mat

In [ ]:
for_mat_0 = for_mat[for_mat.stability==0]
for_mat_1 = for_mat[for_mat.stability==1]
for_mat_1 = for_mat_1[for_mat_1.sum(axis=1) >= 30]
for_mat_1['stability'] = [10 if s==1 else 0 for s in for_mat_1.stability]
for_mat_1[for_mat_1==1] = 2
for_mat_1['stability'] = [1 if s==10 else 0 for s in for_mat_1.stability]
for_mat = pd.concat([for_mat_0, for_mat_1])
for_mat

In [ ]:
# subsample of pairs to be plotted
random.seed(1)
nplot = 25
pairs_toplot = for_mat.reset_index().groupby('stability').pair.apply(lambda x: random.sample(sorted(x), nplot)).explode().to_list()
# Map of values to colors
cs = sns.husl_palette(2)
value_to_color = {0: 'white', 1: cs[0], 2: cs[1]}
# Create colormap and norm
values = list(value_to_color.keys())
colors = [value_to_color[val] for val in values]
cmap = ListedColormap(colors)
norm = BoundaryNorm([v - 0.5 for v in values] + [values[-1] + 0.5], cmap.N)

In [ ]:
for_heatmap = data[data.pair.isin(pairs_toplot)][['day','pair']].drop_duplicates().merge(randstab, on='pair')
for_heatmap['val'] = [2 if st==1 else 1 for st in for_heatmap.stability]
for_heatmap

In [ ]:
fullrange_rdm = pd.DataFrame([d.date() for d in pd.date_range(di, df)], columns=['day'])\
                    .merge(pd.DataFrame(pairs_toplot[:nplot], columns=['pair']), how='cross')
fullrange_rdm['stability'] = 0
fullrange_stb = pd.DataFrame([d.date() for d in pd.date_range(di, df)], columns=['day'])\
                    .merge(pd.DataFrame(pairs_toplot[nplot:], columns=['pair']), how='cross')
fullrange_stb['stability'] = 1
fullrange = pd.concat([fullrange_rdm, fullrange_stb])
for_heatmap = fullrange.merge(for_heatmap, on=['day','pair','stability'], how='left')

In [ ]:
sns.set_theme(style="ticks")

# Function to plot a heatmap inside each facet
def heatmap_func(data, **kwargs):
    pivoted = data.pivot(index="pair", columns="day", values="val")  # Reshape
    #print(pivoted.sum(axis=1))
    sns.heatmap(pivoted.loc[::-1], cbar=False, robust=True, cmap=cmap, norm=norm, square=True, **kwargs)

# Create a FacetGrid, grouping by 'category'
g = sns.FacetGrid(for_heatmap.fillna(0), row="stability", margin_titles=False, height=2, aspect=3., row_order=[1,0])

# Map the custom heatmap function to each facet
g.map_dataframe(heatmap_func)

# Set yticks for all subplots
g.set(yticks=[], xticks=[x+.5 for x in range(len(set(data.day)))], ylabel='device pair', xlabel='day')
g.set_yticklabels([])
g.set_xticklabels([str(d.month).zfill(2)+'/'+str(d.day).zfill(2) if d.weekday()==6 and d.isocalendar().week%2==0\
                   else '' for d in sorted(set(data.day))], rotation=0)

for ax in g.axes.flat:
    #ax.set_title(ax.get_title().split('=')[1][1:])
    ax.tick_params(axis="x", labelbottom=True)

# Modify margin titles
g.set_titles(row_template='')#"{row_name}")#, col_template="{col_name}", size=14, fontweight='bold')

#g.tight_layout()
plt.savefig(f'plots/fig6/stable_random_contacts_example.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/fig6/stable_random_contacts_example.pdf', bbox_inches='tight')
plt.show()

In [ ]:
print([[255*n for n in c] for c in cs])